# BusinessGPT DPO Fine-Tuning: Qwen3.5-2B (abliterated) — preference learning on top of SFT

Trains a second LoRA adapter on top of an existing SFT model, using preference pairs from human pairwise labeling.

**Pipeline position:**

1. SFT (`training.ipynb`) → SFT model on HF
2. **Multi-candidate eval** generates 4 candidates per prompt → `eval/generations_<V>_multi.json`
3. Pairwise labeling (`pairwise_ui_multi` in `businessgpt_bench.ipynb`) → `eval/ratings_<V>_multi.json`
4. `build_dpo_from_multi("<V>")` in bench → `eval/preference_pairs_<V>_multi.jsonl` (in-distribution, all from same SFT model)
5. Upload to Kaggle dataset `businessgpt-eval`
6. **This notebook** → DPO model on HF
7. Eval cell at end → `eval/generations_<DPO_VERSION>.json` for bench comparison

Set `BASE_VERSION` / `DPO_VERSION` / `USE_MULTI_PAIRS` in the config cell before running.

**v14-dpo run notes**: switched from cross-version pairs (off-policy, caused 3× collapse on v13-dpo) to in-distribution v14-self pairs. Hypers relaxed back to standard DPO defaults — tight anchor was a workaround for bad data, no longer needed.

**Reload from HF only:** set `LOAD_DPO_FROM_HF = True`, run **Load DPO from Hugging Face**, then section 7 — skip preference pairs and training (sections 2–6).

**Requirements**: Kaggle GPU T4×2; preference pairs file in attached `businessgpt-eval` dataset.

## 0. Install Dependencies

In [ ]:
%%capture
%pip install vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly
%pip install git+https://github.com/huggingface/transformers.git
%pip install -U trl kagglehub datasets accelerate bitsandbytes peft torchao


In [ ]:
import site, shutil
from pathlib import Path

_paths = list(site.getsitepackages()) + [site.getusersitepackages()]
for _base in _paths:
    _base = Path(_base)
    if not _base.exists():
        continue
    for _pat in ("huggingface_hub", "huggingface_hub-*.dist-info"):
        for _p in _base.glob(_pat):
            print(f"removing stale {_p}")
            shutil.rmtree(_p, ignore_errors=True)


In [ ]:
%pip install --no-cache-dir --force-reinstall git+https://github.com/huggingface/huggingface_hub.git


In [ ]:
# Import sanity check for Save & Run; install cells above should make this pass.
import huggingface_hub
from huggingface_hub.errors import DryRunError
print(f"huggingface_hub={huggingface_hub.__version__}")
print("huggingface_hub DryRunError import OK")


In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)


## 1. Configuration

In [ ]:
# Edit these for each DPO run.
BASE_VERSION = "v15"            # SFT version to start from (also = ref model)
DPO_VERSION  = "v15-dpo"        # output name for the DPO adapter
SUPER_WEIGHT = 1                # multi-candidate data has no super tier; ignored

# Data source: True = in-distribution v<BASE>_multi pairs (recommended).
# False = legacy cross-version preference_pairs.jsonl (off-policy; previously collapsed).
USE_MULTI_PAIRS = True

# Auto-derive size suffix from BASE_VERSION naming convention.
# v11..v14 → 2b, v15+ → 9b. Override _MODEL_SIZE manually if needed.
_MODEL_SIZE = "9b" if int(__import__("re").match(r"v(\d+)", BASE_VERSION).group(1)) >= 15 else "2b"
_BASE_NAME  = f"Huihui-Qwen3.5-{_MODEL_SIZE.upper()}-abliterated"

BASE_MODEL_ID = f"huihui-ai/{_BASE_NAME}"
SFT_REPO  = f"vXofi/businessgpt-{BASE_VERSION}-qwen3.5-{_MODEL_SIZE}"
DPO_REPO  = f"vXofi/businessgpt-{DPO_VERSION}-qwen3.5-{_MODEL_SIZE}"
SAVE_DIR  = f"businessgpt_{DPO_VERSION}_{_MODEL_SIZE}_model"

# Inference-only: set True and run "Load DPO from Hugging Face" after this cell, then jump to section 7 (eval).
# Training cells will raise if you forget and run them with this flag on.
LOAD_DPO_FROM_HF = False

print(f"BASE: {SFT_REPO}  ({BASE_MODEL_ID}, size={_MODEL_SIZE})")
print(f"OUT:  {DPO_REPO}")
print(f"DATA: {'multi (in-distribution)' if USE_MULTI_PAIRS else 'cross-version (off-policy)'}")

## Load DPO from Hugging Face (inference-only)

If you **already pushed** the DPO adapter to Hugging Face, you can skip preference pairs and training (sections **2–6**):

1. Set `LOAD_DPO_FROM_HF = True` in the config cell (prevents accidentally running training cells).
2. Run install → login → config → **the next code cell** → section **7** (eval).

This reconstructs the stack the same way as training: **base → merge SFT → attach DPO LoRA** from `DPO_REPO`.

In [ ]:
if not LOAD_DPO_FROM_HF:
    print("Skipping HF DPO load (LOAD_DPO_FROM_HF=False). Run sections 2–6 to train, or set the flag and re-run this cell.")
else:
    import os
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
    from peft import PeftModel
    import torch

    MAX_SEQ_LENGTH = 1536

    try:
        _base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"AutoModelForCausalLM failed ({e}), trying AutoModel...")
        _base = AutoModel.from_pretrained(
            BASE_MODEL_ID,
            dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )

    if next(_base.parameters()).dtype not in (torch.float16, torch.float32):
        _base = _base.half()

    _sft = PeftModel.from_pretrained(_base, SFT_REPO)
    _merged = _sft.merge_and_unload()
    del _sft, _base

    model = PeftModel.from_pretrained(_merged, DPO_REPO)
    print(f"Loaded DPO LoRA on merged SFT: {DPO_REPO} (SFT: {SFT_REPO})")

    tokenizer = AutoTokenizer.from_pretrained(DPO_REPO, trust_remote_code=True)
    CHAT_TEMPLATE_NO_THINK = (
        "{%- for message in messages %}"
        "<|im_start|>{{ message.role }}\n"
        "{{ message.content }}<|im_end|>\n"
        "{%- endfor %}"
        "{%- if add_generation_prompt %}"
        "<|im_start|>assistant\n"
        "{%- endif %}"
    )
    tokenizer.chat_template = CHAT_TEMPLATE_NO_THINK
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Model: {model.__class__.__name__}, dtype={next(model.parameters()).dtype}")

    # So section 7 (eval) can upload generations without running the push cell.
    from huggingface_hub import HfApi
    HF_REPO = DPO_REPO
    api = HfApi()

## 2. Load Preference Pairs

In [ ]:
import json
import os
import glob as _glob

# Pick filename based on data source.
PAIRS_FILE = (
    f"preference_pairs_{BASE_VERSION}_multi.jsonl"
    if USE_MULTI_PAIRS
    else "preference_pairs.jsonl"
)

_candidates = [
    f"/kaggle/input/businessgpt-eval/{PAIRS_FILE}",
    f"/kaggle/input/businessgpt-eval/eval/{PAIRS_FILE}",
    f"eval/{PAIRS_FILE}",
    PAIRS_FILE,
]
_pairs_path = next((p for p in _candidates if os.path.isfile(p)), None)

if _pairs_path is None:
    _matches = _glob.glob(f"/kaggle/input/**/{PAIRS_FILE}", recursive=True)
    if _matches:
        _pairs_path = _matches[0]
        print(f"Found {PAIRS_FILE} via glob: {_pairs_path}")

if _pairs_path is None:
    raise FileNotFoundError(
        f"{PAIRS_FILE} not found. For multi pairs: run pairwise_ui_multi + "
        f"build_dpo_from_multi in the bench notebook, then version the businessgpt-eval "
        f"Kaggle dataset (`kaggle datasets version -p eval`)."
    )

with open(_pairs_path, encoding="utf-8") as f:
    pairs = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(pairs)} preference pairs from {_pairs_path}")

# Stats
from collections import Counter
print(f"  by category: {Counter(p['category'] for p in pairs)}")
print(f"  by tier:     {Counter(p.get('tier', 'normal') for p in pairs)}")

# Apply super-tier weighting by replication.
weighted = []
for p in pairs:
    weighted.append(p)
    if p.get("tier") == "super" and SUPER_WEIGHT > 1:
        for _ in range(SUPER_WEIGHT - 1):
            weighted.append(p)
print(f"After super-weighting (×{SUPER_WEIGHT}): {len(weighted)} pairs")

## 3. Load Base Model + SFT Adapter (merge)

In [ ]:
import os
# Set before any torch/cuda init to reduce fragmentation (see PyTorch CUDA memory notes).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if LOAD_DPO_FROM_HF:
    raise RuntimeError(
        "LOAD_DPO_FROM_HF is True: skip this cell. Use 'Load DPO from Hugging Face' above, then section 7 (eval)."
    )

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
from peft import PeftModel
import torch

# DPO is heavy (precompute ref + policy + KV cache). Lower MAX_SEQ_LENGTH on bigger models.
# 2B: 1536 was fine.   9B: 1024 needed — 9B fp32 = 36 GB which doesn't fit T4×2 (30 GB).
MAX_SEQ_LENGTH = 1024 if _MODEL_SIZE == "9b" else 1536

# Precision strategy:
#   2B: fp32 worked (8 GB total, fits comfortably). Required because Qwen3.5 hybrid
#       linear_attn layers NaN under fp16 precompute_ref_log_probs.
#   9B: fp32 doesn't fit (36 GB > 30 GB on T4×2). Fall back to fp16 — but 9B may
#       have the same hybrid-layer NaN issue. If precompute_ref hits NaN, plan B:
#       QLoRA (4-bit base + fp16 LoRA via bitsandbytes BitsAndBytesConfig).
DPO_DTYPE = torch.float16 if _MODEL_SIZE == "9b" else torch.float32

try:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=DPO_DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )
except Exception as e:
    print(f"AutoModelForCausalLM failed ({e}), trying AutoModel...")
    base_model = AutoModel.from_pretrained(
        BASE_MODEL_ID,
        dtype=DPO_DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )

# Force the requested dtype (model may come down in native bf16).
if next(base_model.parameters()).dtype != DPO_DTYPE:
    print(f"Casting from {next(base_model.parameters()).dtype} to {DPO_DTYPE}...")
    if DPO_DTYPE == torch.float32:
        base_model = base_model.float()
    elif DPO_DTYPE == torch.float16:
        base_model = base_model.half()

# Apply SFT adapter on top, then merge into the base. The merged model becomes
# the reference distribution for DPO; the new LoRA (next cell) is what DPO trains.
sft = PeftModel.from_pretrained(base_model, SFT_REPO)
model = sft.merge_and_unload()
print(f"Merged SFT adapter from {SFT_REPO} into base.")

tokenizer = AutoTokenizer.from_pretrained(SFT_REPO, trust_remote_code=True)
CHAT_TEMPLATE_NO_THINK = (
    "{%- for message in messages %}"
    "<|im_start|>{{ message.role }}\n"
    "{{ message.content }}<|im_end|>\n"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}"
    "<|im_start|>assistant\n"
    "{%- endif %}"
)
tokenizer.chat_template = CHAT_TEMPLATE_NO_THINK
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model: {model.__class__.__name__}, dtype={next(model.parameters()).dtype}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")

## 4. Apply DPO LoRA

In [ ]:
from peft import LoraConfig, get_peft_model

if LOAD_DPO_FROM_HF:
    raise RuntimeError("LOAD_DPO_FROM_HF is True: skip this cell (DPO adapter already loaded from HF).")

# Qwen3.5 is hybrid: half the layers use full attention (q/k/v/o_proj), half use
# **linear attention** (linear_attn.in_proj_qkv / linear_attn.out_proj). The linear-attn
# layers carry recurrent state that goes NaN under fp16 backward over long sequences —
# the previous DPO run blew up with all 372 corrupt params inside linear_attn.
# Solution: restrict LoRA to standard attention + MLP only. SFT v13 trained "all-linear"
# (covers linear_attn too); those weights stay baked in via merge_and_unload, so we don't
# lose anything — DPO just nudges the stable layers.
DPO_TARGET_MODULES = [
    # full self-attention
    "q_proj", "k_proj", "v_proj", "o_proj",
    # MLP
    "gate_proj", "up_proj", "down_proj",
]

dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=DPO_TARGET_MODULES,
    lora_dropout=0.05,
    use_dora=False,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, dpo_lora_config)
model.gradient_checkpointing_enable()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"DPO LoRA: {trainable:,} / {total:,} ({trainable/total*100:.2f}%) trainable")
print(f"Targets: {DPO_TARGET_MODULES}  (linear_attn intentionally excluded)")

## 5. DPO Trainer

In [ ]:
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

if LOAD_DPO_FROM_HF:
    raise RuntimeError("LOAD_DPO_FROM_HF is True: skip DPO trainer (dataset + training).")
import random as _random

_random.seed(42)
_random.shuffle(weighted)
val_size = max(20, int(len(weighted) * 0.05))
val_pairs = weighted[:val_size]
train_pairs = weighted[val_size:]

train_ds = Dataset.from_list(train_pairs)
val_ds   = Dataset.from_list(val_pairs)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

# --- Sanity-check pairs BEFORE expensive training ---
_n_identical, _n_short = 0, 0
for p in train_pairs:
    c = p["chosen"][-1]["content"].strip()
    r = p["rejected"][-1]["content"].strip()
    if c == r:
        _n_identical += 1
    if len(c) < 2 or len(r) < 2:
        _n_short += 1
print(f"Sanity: identical chosen/rejected = {_n_identical}, very-short responses = {_n_short}")
assert _n_identical < len(train_pairs) * 0.05, \
    "Too many identical chosen/rejected pairs — DPO has no signal."

# Length-bias diagnostic — printed for awareness, doesn't block.
_chosen_lens = [len(p["chosen"][-1]["content"]) for p in train_pairs]
_rejected_lens = [len(p["rejected"][-1]["content"]) for p in train_pairs]
_ratio = sum(_chosen_lens) / max(sum(_rejected_lens), 1)
print(f"Length: chosen mean={sum(_chosen_lens)/len(_chosen_lens):.1f}, "
      f"rejected mean={sum(_rejected_lens)/len(_rejected_lens):.1f}, "
      f"ratio={_ratio:.2f}  (closer to 1.0 = less length bias)")

NUM_EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
STEPS_PER_EPOCH = max(1, len(train_ds) // EFFECTIVE_BATCH)
EVAL_STEPS = max(1, STEPS_PER_EPOCH // 4)

# Stability config v5 (in-distribution data + standard DPO hypers):
# Previous v3-DPO runs all collapsed because data was 100% off-policy (v11/v12 pairs vs v13 ref).
# We tightened beta/lr to compensate, which prevented learning.
# v14-DPO uses in-distribution pairs (v14_multi self-comparisons), so we relax back to standard:
#   - beta 0.8 → 0.1   (default DPO; KL anchor only mild — let the model actually move)
#   - lr 5e-8 → 5e-7   (two orders of magnitude up; meaningful gradient steps)
#   - epochs 1 → 2     (data is smaller; one full pass is undertrained)
# Architectural safeguards (fp32, exclude linear_attn, precompute_ref) STAY — those are about
# the Qwen3.5 hybrid architecture on T4, unrelated to data quality.
dpo_args = DPOConfig(
    output_dir="dpo_outputs",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=5e-7,
    beta=0.1,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    max_length=MAX_SEQ_LENGTH,
    truncation_mode="keep_start",
    precompute_ref_log_probs=True,
    precompute_ref_batch_size=1,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    seed=3407,
    report_to="none",
)


# --- Stability callback: catch NaN/Inf loss + reward-margin explosion early ---
# Reward margin = β · (logπ(chosen) - logπ(rejected) - logπ_ref(chosen) + logπ_ref(rejected)).
# If it grows monotonically without bound, the policy is sprinting away from ref → soft collapse.
# Threshold 50 is empirical: typical healthy DPO sees margins 1-10. >50 means runaway.
from transformers import TrainerCallback
import math

class _StabilityAbortCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        loss = logs.get("loss")
        if loss is not None and (math.isnan(loss) or math.isinf(loss)):
            print(f"\n⚠️ NaN/Inf training loss at step {state.global_step} — aborting.")
            control.should_training_stop = True
            return
        margin = logs.get("rewards/margins")
        if margin is not None and abs(margin) > 50:
            print(f"\n⚠️ rewards/margins = {margin:.2f} (>50) at step {state.global_step} — runaway, aborting.")
            control.should_training_stop = True


trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    callbacks=[_StabilityAbortCallback()],
)

print(f"Training config:")
print(f"  Train pairs: {len(train_ds)} (effective batch {EFFECTIVE_BATCH})")
print(f"  Steps/epoch: {STEPS_PER_EPOCH}, total: {STEPS_PER_EPOCH * NUM_EPOCHS}")
print(f"  beta={dpo_args.beta}  lr={dpo_args.learning_rate}  super_weight={SUPER_WEIGHT}")
print(f"  precision: fp32 (fp16=False, bf16=False)")

In [ ]:
if LOAD_DPO_FROM_HF:
    raise RuntimeError("LOAD_DPO_FROM_HF is True: skip training.")
trainer_stats = trainer.train()
print(f"\nDPO training complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Final loss:  {trainer_stats.training_loss:.4f}")

# --- Post-training NaN check ---
# If weights NaN'd, generation will collapse to "!!!!!!!". Catch it here, not at eval.
import torch
nan_params = [n for n, p in model.named_parameters() if torch.isnan(p).any()]
inf_params = [n for n, p in model.named_parameters() if torch.isinf(p).any()]
if nan_params or inf_params:
    print(f"\n⚠️ MODEL CORRUPTED:")
    print(f"  {len(nan_params)} params contain NaN: {nan_params[:3]}{'...' if len(nan_params)>3 else ''}")
    print(f"  {len(inf_params)} params contain Inf: {inf_params[:3]}{'...' if len(inf_params)>3 else ''}")
    print("  → Lower learning_rate further (5e-8 or 1e-8), raise beta to 0.7, or shrink max_length.")
    raise RuntimeError("DPO produced NaN/Inf weights — do NOT push, retrain with safer hyperparams.")
else:
    print(f"  Sanity: no NaN/Inf in trainable params.")

# Quick generation smoke test — must produce something, not "!!!!".
import random as _r
_r.seed(0)
_test_prompt = "Некит Русанов: пиздец как дела?"
_inp = tokenizer(
    f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{_test_prompt}<|im_end|>\n<|im_start|>assistant\n",
    return_tensors="pt", add_special_tokens=False,
).to(next(model.parameters()).device)
with torch.no_grad():
    _out = model.generate(**_inp, max_new_tokens=30, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
_smoke = tokenizer.decode(_out[0][_inp["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"\nSmoke generation:\n  prompt: {_test_prompt}\n  response: {_smoke!r}")
if _smoke.strip() in ("", "!", "!!", "!!!") or _smoke.strip().count("!") > 5:
    raise RuntimeError("Smoke test failed — model collapsed to dead token. Do not push.")

## 6. Save & Push

In [ ]:
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
print(f"DPO adapter saved to {SAVE_DIR}/")

import os
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f"{SAVE_DIR}/{f}")
    if size > 1024 * 1024:
        print(f"  {f}: {size/1024/1024:.1f} MB")
    else:
        print(f"  {f}: {size/1024:.1f} KB")

In [ ]:
from huggingface_hub import HfApi

HF_REPO = DPO_REPO
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=HF_REPO,
    commit_message=f"Upload BusinessGPT {DPO_VERSION} DPO adapter (base={BASE_VERSION}, beta=0.1, lr=5e-7, in-distribution multi pairs)",
)
print(f"Pushed to https://huggingface.co/{HF_REPO}")

## 7. Eval: generate on golden_prompts set

Same eval cell as `training.ipynb` — produces `eval/generations_<DPO_VERSION>.json`
that the bench notebook can compare against the SFT base via
`pairwise_ui("v13", "v13-dpo")` and `guardrails_table(...)`.

In [ ]:
import io
import contextlib
import re as _re

EVAL_DIR = "/kaggle/working/eval"
os.makedirs(EVAL_DIR, exist_ok=True)
UPLOAD_EVAL_TO_HF = False  # private eval generations must not be pushed to model repos

def _block_private_eval_uploads():
    if not UPLOAD_EVAL_TO_HF and "api" in globals():
        def _blocked_upload_file(*args, **kwargs):
            raise RuntimeError("UPLOAD_EVAL_TO_HF=False; private eval generations stay local/Kaggle output")
        api.upload_file = _blocked_upload_file

_block_private_eval_uploads()
VERSION = DPO_VERSION

# Locate golden_prompts.json
_candidates = [
    "/kaggle/input/businessgpt-eval/golden_prompts.json",
    "/kaggle/input/businessgpt-eval/eval/golden_prompts.json",
    "eval/golden_prompts.json",
    "golden_prompts.json",
]
_golden_path = next((p for p in _candidates if os.path.isfile(p)), None)
if _golden_path is None:
    _matches = _glob.glob("/kaggle/input/**/golden_prompts.json", recursive=True)
    if _matches:
        _golden_path = _matches[0]
if _golden_path is None:
    raise FileNotFoundError("golden_prompts.json not found in Kaggle input or local eval/. Refusing HF fallback for private eval data.")

with open(_golden_path, encoding="utf-8") as f:
    golden = json.load(f)
print(f"Loaded {len(golden)} prompts from {_golden_path}")

# Inference helper (mirror of training.ipynb's chat())
SYSTEM_PROMPT_INFER = (
    "Ты BusinessGPT. Пиши как студент в мессенджере: коротко, дерзко, ахуевше, "
    "по-пидорски. Часто вставляй слова-паразиты: бля, нах, блять, ёпт, пиздец."
)

def _format_chat(messages):
    return "\n".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>" for m in messages)

_im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
_im_end_id   = tokenizer.convert_tokens_to_ids("<|im_end|>")
_vocab_sz = getattr(tokenizer, "vocab_size", None) or len(tokenizer)


def _valid_tid(tid):
    if tid is None or not isinstance(tid, int):
        return False
    if tid == tokenizer.unk_token_id:
        return False
    return 0 <= tid < _vocab_sz


_eos_ids = [x for x in (_im_end_id, _im_start_id) if _valid_tid(x)]
if not _eos_ids:
    _eos_ids = [int(tokenizer.eos_token_id)]
# Prefer a single EOS id for GenerationConfig (fewer edge cases than multi-eos on some stacks).
_eos_for_generate = _eos_ids[0] if len(_eos_ids) == 1 else _eos_ids

_pad_id = tokenizer.pad_token_id
if _pad_id is None:
    _pad_id = tokenizer.eos_token_id

# `torch.multinomial` can hit a CUDA device-side assert; after that the GPU context stays broken until
# process restart — a second `generate` on GPU is not a reliable fallback. Use greedy (no sampling) for batch eval.
USE_SAMPLED_EVAL = False
print(f"Eval generate: do_sample={USE_SAMPLED_EVAL}  eos_token_id={_eos_for_generate!r}")

_input_device = next(model.parameters()).device
model.eval()
torch.manual_seed(42)

_gen_ctx = (
    torch.amp.autocast("cuda", enabled=False)
    if _input_device.type == "cuda"
    else contextlib.nullcontext()
)


def chat(messages_history, max_new_tokens=200):
    if messages_history and isinstance(messages_history[0], str):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_INFER},
            {"role": "user",   "content": "\n".join(messages_history)},
        ]
    else:
        messages = [{"role": "system", "content": SYSTEM_PROMPT_INFER}] + list(messages_history)
    input_text = _format_chat(messages) + "\n<|im_start|>assistant\n"
    inputs = tokenizer(input_text, return_tensors="pt", add_special_tokens=False).to(_input_device)
    _common = dict(
        max_new_tokens=max_new_tokens,
        eos_token_id=_eos_for_generate,
        pad_token_id=_pad_id,
    )
    with torch.no_grad():
        with _gen_ctx:
            if USE_SAMPLED_EVAL:
                out = model.generate(
                    **inputs,
                    do_sample=True,
                    temperature=0.8,
                    top_p=0.9,
                    repetition_penalty=1.0,
                    renormalize_logits=True,
                    **_common,
                )
            else:
                out = model.generate(**inputs, do_sample=False, **_common)
    g = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    g = _re.sub(r"<think>.*?</think>\s*", "", g, flags=_re.DOTALL).strip()
    g = _re.sub(r"<\|im_start\|>.*", "", g, flags=_re.DOTALL)
    g = _re.sub(r"<\|im_end\|>", "", g)
    g = _re.sub(r"^<bot>\s*", "", g).strip()
    return g

SAMPLING = {
    "do_sample": USE_SAMPLED_EVAL,
    "temperature": 0.8,
    "top_p": 0.9,
    "repetition_penalty": 1.0,
    "max_new_tokens": 200,
    "note": "Batch eval uses greedy (do_sample=False) unless USE_SAMPLED_EVAL=True; CUDA multinomial asserts poison the device.",
}

generations = []
for i, p in enumerate(golden):
    with contextlib.redirect_stdout(io.StringIO()):
        response = chat(p["context"], max_new_tokens=SAMPLING["max_new_tokens"])
    generations.append({
        "id": p["id"], "category": p["category"], "held_out": p.get("held_out", False),
        "context": p["context"], "response": response,
        "version": VERSION, "sampling_params": SAMPLING, "seed": 42,
    })
    if (i + 1) % 25 == 0 or (i + 1) == len(golden):
        print(f"  {i+1}/{len(golden)}")

out_path = os.path.join(EVAL_DIR, f"generations_{VERSION}.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(generations, f, ensure_ascii=False, indent=1)
print(f"\nSaved {len(generations)} generations to {out_path}")

try:
    api.upload_file(
        path_or_fileobj=out_path,
        path_in_repo=f"eval/generations_{VERSION}.json",
        repo_id=HF_REPO,
        commit_message=f"Upload eval generations ({VERSION})",
    )
    print(f"Uploaded to https://huggingface.co/{HF_REPO}/blob/main/eval/generations_{VERSION}.json")
except Exception as e:
    print(f"HF upload failed (non-fatal): {e}")